# 04 - Pandas Foundations: Series, DataFrame, and the Index

**Prerequisite:** notebooks 01-03 (Pandas is built on NumPy).
**This notebook:** the two core objects, dtypes, reading data, and every way to select and filter rows - mapped to SQL throughout.

---

## The mental model

| Pandas | SQL | Java |
|---|---|---|
| `DataFrame` | a table / result set | `List<Order>` or a `ResultSet` |
| `Series` | one column | `double[]` with labels, or `List<Double>` |
| `Index` | primary key / row labels | the `Map` key, or row position |
| `dtype` | column type | field type |
| `df[df.x > 5]` | `WHERE x > 5` | `stream().filter(...)` |
| `df.groupby("k").sum()` | `GROUP BY k` | `Collectors.groupingBy(...)` |
| `df.merge(other)` | `JOIN` | a manual hash join |

The one idea with no SQL or Java equivalent is the **Index**. A SQL table has no inherent row order or row label; a DataFrame always has both. Most Pandas confusion traces back to the index, so it gets its own section.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

DATA = next(p for p in [Path("../data"), Path("data"), Path("study/data")] if p.exists())
print("pandas", pd.__version__, "| numpy", np.__version__)
print("data folder:", DATA.resolve())

## 1. Series - a labelled column

A `Series` is a NumPy array plus an index. The index is what makes Pandas different from NumPy.

In [ ]:
s = pd.Series([10, 20, 30, 40], name="amount")
print(s)
print()
print("values (a NumPy array):", s.values)
print("index                 :", list(s.index))
print("dtype                 :", s.dtype)

In [ ]:
# A custom index turns the Series into an ordered Map<String, Double>
rev = pd.Series([250.0, 310.5, 180.0], index=["kolkata", "pune", "delhi"], name="revenue")
print(rev)
print()
print("rev['pune']      ->", rev["pune"], "  (label lookup, like a Map.get)")
print("rev.iloc[1]      ->", rev.iloc[1], "  (position lookup, like an array index)")

### Automatic alignment - the behaviour that surprises everyone

Arithmetic between two Series aligns on the **index**, not on position. It is a full outer join on the labels.

In [ ]:
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["z", "y", "x"])      # same labels, different order

print("a + b (aligned by label, not position):")
print(a + b)
print()
c = pd.Series([100, 200], index=["y", "w"])
print("mismatched labels produce NaN, like an outer join:")
print(a + c)

## 2. DataFrame - a table

A `DataFrame` is a dict of Series that all share one index. Columns can have different dtypes; a NumPy 2-D array cannot.

In [ ]:
df = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "customer": ["acme", "globex", "acme", "initech"],
    "amount":   [250.5, 99.99, 1200.0, 45.0],
    "paid":     [True, False, True, True],
})
print(df)
print()
print(df.dtypes)

In [ ]:
# Other ways to build one
print("from a list of dicts (like a JSON payload):")
print(pd.DataFrame([{"a": 1, "b": 2}, {"a": 3, "b": 4}]))
print()
print("from a NumPy array with explicit columns:")
print(pd.DataFrame(np.arange(6).reshape(2, 3), columns=["c1", "c2", "c3"]))
print()
print("from a list of tuples (like JDBC rows):")
print(pd.DataFrame([(1, "a"), (2, "b")], columns=["id", "code"]))

## 3. Reading data

`read_csv` is the workhorse. It has ~50 parameters; these are the ones that matter.

In [ ]:
orders = pd.read_csv(
    DATA / "orders.csv",
    parse_dates=["order_ts"],        # parse to datetime64 instead of leaving as text
)
customers = pd.read_csv(DATA / "customers.csv", parse_dates=["signup_date"])
items     = pd.read_csv(DATA / "order_items.csv")

print(orders.shape, customers.shape, items.shape)
orders.head()

| `read_csv` argument | What it does |
|---|---|
| `parse_dates=[...]` | convert columns to `datetime64` |
| `dtype={"id": "int32"}` | force column types, avoids a second pass and saves memory |
| `usecols=[...]` | read only some columns - the cheapest optimisation there is |
| `nrows=1000` | read a sample first |
| `chunksize=100_000` | returns an iterator of DataFrames, for files bigger than RAM |
| `na_values=["NA", "-"]` | extra strings to treat as missing |
| `sep`, `encoding`, `compression` | tab-separated, latin-1, gzip, etc. |
| `index_col="order_id"` | use a column as the index directly |

Other readers you will use: `pd.read_parquet` (columnar, typed, compressed - prefer it over CSV for anything you control), `pd.read_json`, `pd.read_sql(query, conn)`, `pd.read_excel`.

In [ ]:
# Reading straight from a database - this is how Pandas meets your JDBC world
import sqlite3

conn = sqlite3.connect(":memory:")
orders.to_sql("orders", conn, index=False, if_exists="replace")

top = pd.read_sql("SELECT channel, COUNT(*) AS n, AVG(amount) AS avg_amt FROM orders GROUP BY channel", conn)
print(top)
conn.close()

## 4. Inspecting a DataFrame

These four are your first four commands on any unfamiliar dataset.

In [ ]:
orders.info()

In [ ]:
orders.describe()

In [ ]:
print("shape   :", orders.shape)
print("columns :", list(orders.columns))
print()
print("memory per column:")
print(orders.memory_usage(deep=True))

In [ ]:
print(orders["status"].value_counts())
print()
print("normalised (share of rows):")
print(orders["status"].value_counts(normalize=True).round(3))
print()
print("nunique per column:")
print(orders.nunique())

## 5. dtypes in Pandas

| dtype | Notes |
|---|---|
| `int64`, `float64` | NumPy-backed. `int64` **cannot** hold missing values |
| `bool` | NumPy-backed, no missing values |
| `str` | text. In Pandas 3.x this is the default for string columns |
| `datetime64[ns]` | timestamps; may carry a timezone (`datetime64[ns, Asia/Kolkata]`) |
| `timedelta64[ns]` | durations |
| `category` | a dictionary-encoded column - huge memory win for low-cardinality text |
| `Int64`, `Float64`, `boolean` (capital letter) | **nullable** extension types that support `pd.NA` |

The classic trap: an `int64` column with one missing value is silently promoted to `float64`, because NumPy integers have no NaN. The nullable `Int64` dtype exists to fix exactly that.

In [ ]:
plain = pd.Series([1, 2, None])
print("plain :", plain.dtype, "->", [str(v) for v in plain])
print("        the ints became floats because NaN is a float")
print()
nullable = pd.Series([1, 2, None], dtype="Int64")
print("Int64 :", nullable.dtype, "->", [str(v) for v in nullable])
print("        stays integer, missing value is pd.NA")

In [ ]:
# category: massive memory saving on repeated text
print("status as str     :", orders["status"].memory_usage(deep=True), "bytes")
print("status as category:", orders["status"].astype("category").memory_usage(deep=True), "bytes")
print()
cat = orders["status"].astype("category")
print("categories stored once:", list(cat.cat.categories))
print("values stored as small integer codes:", cat.cat.codes.head(5).tolist())

## 6. The Index

Every DataFrame has a row index. By default it is `RangeIndex(0, n)` - positional and meaningless. Setting a real index turns the DataFrame into a lookup table.

In [ ]:
ind = orders.set_index("order_id")
print(ind.head(3))
print()
print("label lookup, O(1)-ish via a hash table:")
print(ind.loc[9415])

In [ ]:
print("reset_index puts it back as a column:")
print(ind.reset_index().head(2))
print()
print("is the index unique?", ind.index.is_unique)
print("is it sorted?      ", ind.index.is_monotonic_increasing)

> **Practical advice:** set an index when you will do repeated label lookups, joins on that key, or time-based slicing. Otherwise leave the default `RangeIndex` and use columns. A non-unique index is legal but causes surprising result sizes on joins and lookups - check `index.is_unique` when debugging.

## 7. Selecting columns and rows

This is the part worth memorising.

| You want | Write |
|---|---|
| one column as a Series | `df["amount"]` |
| one column as a DataFrame | `df[["amount"]]` |
| several columns | `df[["a", "b"]]` |
| rows by **label** | `df.loc[label]`, `df.loc[l1:l2]` (**end inclusive**) |
| rows by **position** | `df.iloc[0]`, `df.iloc[0:5]` (end exclusive) |
| a cell by label | `df.at[label, "col"]` (fastest) |
| a cell by position | `df.iat[0, 2]` |
| rows + columns | `df.loc[mask, ["a", "b"]]` |
| filter rows | `df[mask]` or `df.loc[mask]` |

`.loc` is **label-based**, `.iloc` is **position-based**. With the default `RangeIndex` the labels happen to be 0,1,2..., which hides the difference - until you filter or sort and it bites.

In [ ]:
small = orders.head(6).copy()
print(small[["order_id", "status", "amount"]])
print()
print("loc with a label slice INCLUDES the end label:")
print(small.loc[1:3, ["order_id", "amount"]])
print()
print("iloc with a position slice EXCLUDES the end:")
print(small.iloc[1:3, [0, 5]])

In [ ]:
# Why the difference matters: after filtering, labels are no longer 0..n
filtered = orders[orders["amount"] > 3000].head(4)
print("index labels after filtering:", list(filtered.index))
print()
print("iloc[0] - first ROW of the result:")
print(filtered.iloc[0][["order_id", "amount"]].to_dict())
print()
print("loc[<first label>] - the same row, but addressed by its label:")
print(filtered.loc[filtered.index[0]][["order_id", "amount"]].to_dict())
print()
try:
    filtered.loc[0]
except KeyError:
    print("filtered.loc[0] -> KeyError: label 0 no longer exists in this result.")
    print("This is the bug people hit when they use loc where they meant iloc.")

## 8. Filtering - your `WHERE` clause

In [ ]:
# Single condition
big = orders[orders["amount"] > 3000]
print("rows with amount > 3000:", len(big))

# Multiple conditions: & | ~ with parentheses (exactly as in NumPy)
web_big = orders[(orders["channel"] == "web") & (orders["amount"] > 2000)]
print("web AND amount > 2000  :", len(web_big))

# IN (...)
subset = orders[orders["status"].isin(["SHIPPED", "DELIVERED"])]
print("status IN (...)        :", len(subset))

# BETWEEN
mid = orders[orders["amount"].between(1000, 2000)]
print("amount BETWEEN 1000 AND 2000:", len(mid))

# IS NULL / IS NOT NULL
print("amount IS NULL         :", orders["amount"].isna().sum())
print("amount IS NOT NULL     :", orders["amount"].notna().sum())

In [ ]:
# LIKE - via the .str accessor, which vectorizes string ops
print("customers whose name ends in 7:", customers[customers["name"].str.endswith("7")]["name"].tolist())
print("city LIKE '%a%'               :", customers[customers["city"].str.contains("a", na=False)].shape[0], "rows")
print()
print("Some .str methods:", [m for m in ["lower", "upper", "strip", "replace", "split", "len", "startswith", "extract", "pad", "zfill"]])

In [ ]:
# query(): a string expression. Reads like SQL, and avoids repeating the DataFrame name.
q = orders.query("channel == 'web' and amount > 2000 and status != 'CANCELLED'")
print("query() rows:", len(q))

threshold = 2500
print("with a Python variable (@ prefix):", len(orders.query("amount > @threshold")))

> `query()` is easier to read for long predicates and can be faster on large frames (it can use the `numexpr` engine). It is slower on small frames because of parsing overhead. Use whichever is clearer.

## 9. Creating and changing columns

In [ ]:
o = orders.copy()

# Direct assignment
o["net_amount"] = o["amount"] * (1 - o["discount_pct"] / 100)

# Conditional column: np.where is CASE WHEN
o["is_large"] = np.where(o["amount"] > 2000, "large", "small")

# Multi-branch: pd.cut is CASE WHEN with ranges
o["size_band"] = pd.cut(o["amount"],
                        bins=[0, 500, 1500, 3000, np.inf],
                        labels=["XS", "S", "M", "L"])

print(o[["amount", "discount_pct", "net_amount", "is_large", "size_band"]].head())

In [ ]:
# assign(): returns a NEW frame, so you can chain. Good for pipelines.
result = (orders
          .assign(net=lambda d: d["amount"] * (1 - d["discount_pct"] / 100))
          .assign(year=lambda d: d["order_ts"].dt.year,
                  month=lambda d: d["order_ts"].dt.month)
          .loc[lambda d: d["net"] > 1000]
          )
print(result[["order_id", "net", "year", "month"]].head())
print("rows:", len(result))

In [ ]:
# Renaming and dropping
renamed = orders.rename(columns={"order_ts": "created_at", "amount": "gross_amount"})
print(list(renamed.columns))
print()
print("drop columns:", list(orders.drop(columns=["discount_pct", "channel"]).columns))
print("drop rows   :", orders.drop(index=[0, 1]).shape)

## 10. Sorting

In [ ]:
print(orders.sort_values("amount", ascending=False).head(3)[["order_id", "amount"]])
print()
print("multi-column, mixed direction (ORDER BY channel ASC, amount DESC):")
print(orders.sort_values(["channel", "amount"], ascending=[True, False]).head(4)[["channel", "amount"]])
print()
print("nlargest is faster than sort+head for a top-N query:")
print(orders.nlargest(3, "amount")[["order_id", "amount"]])

> `nlargest(n, col)` uses a partial selection, not a full sort. It is the Pandas equivalent of `ORDER BY x DESC LIMIT n` with a good plan.

## 11. Missing data

Pandas missing markers: `np.nan` (float columns), `pd.NaT` (datetimes), `pd.NA` (nullable extension types). All behave like SQL `NULL` in aggregates - they are skipped.

In [ ]:
print("nulls per column:")
print(orders.isna().sum())
print()
print("rows with ANY null:", orders.isna().any(axis=1).sum())
print("SUM ignores nulls, exactly like SQL:", round(orders['amount'].sum(), 2))
print("mean over non-null only          :", round(orders['amount'].mean(), 2))

In [ ]:
x = orders[["order_id", "amount"]].head(8).copy()
x.loc[2, "amount"] = np.nan
x.loc[5, "amount"] = np.nan
print(x.to_string(index=False))
print()
print("fillna(0)        :", x["amount"].fillna(0).round(1).tolist())
print("fillna(mean)     :", x["amount"].fillna(x["amount"].mean()).round(1).tolist())
print("ffill (carry fwd):", x["amount"].ffill().round(1).tolist())
print("bfill (carry back):", x["amount"].bfill().round(1).tolist())
print("interpolate      :", x["amount"].interpolate().round(1).tolist())
print()
print("dropna rows:", len(x.dropna()), "of", len(x))
print("dropna with a threshold (keep rows with >=2 non-null):", len(x.dropna(thresh=2)))

## 12. Writing data out

In [ ]:
import tempfile, os
tmp = tempfile.mkdtemp()

orders.head(20).to_csv(os.path.join(tmp, "sample.csv"), index=False)     # index=False almost always
orders.head(20).to_json(os.path.join(tmp, "sample.json"), orient="records", date_format="iso")

print(os.listdir(tmp))
print()
print(open(os.path.join(tmp, "sample.csv")).read().split(chr(10))[0])

For anything you control, prefer **Parquet** over CSV: `df.to_parquet("f.parquet")`. It keeps dtypes, compresses 5-10x better, and supports column pruning on read (`pd.read_parquet(f, columns=["a","b"])`). It needs the `pyarrow` package installed. CSV has no types - every read is a guess.

## 13. Exercises

**E1.** How many orders were placed through the `mobile` channel with `amount` above 1500 and status not `CANCELLED`? Write it two ways: with boolean masks and with `query()`.

**E2.** Add a column `effective_amount` = `amount` after discount, with missing amounts treated as 0.

**E3.** Return the 5 customers who signed up most recently, showing name, city and signup date.

**E4.** Which city has the most customers? Handle the missing city value sensibly.

**E5.** Explain why `orders.loc[0:2]` and `orders.iloc[0:2]` return different numbers of rows.

In [ ]:
# Solutions
m = orders[(orders["channel"] == "mobile") & (orders["amount"] > 1500) & (orders["status"] != "CANCELLED")]
q = orders.query("channel == 'mobile' and amount > 1500 and status != 'CANCELLED'")
print("E1:", len(m), "==", len(q))
print()

e2 = orders.assign(effective_amount=lambda d: d["amount"].fillna(0) * (1 - d["discount_pct"] / 100))
print("E2:", e2["effective_amount"].head(3).round(2).tolist())
print()

print("E3:")
print(customers.nlargest(5, "signup_date")[["name", "city", "signup_date"]].to_string(index=False))
print()

print("E4:")
print(customers["city"].value_counts(dropna=False))
print()
print("E5: loc[0:2] is a LABEL slice and includes label 2 -> 3 rows;")
print("    iloc[0:2] is a POSITION slice and excludes position 2 -> 2 rows.")
print("   ", len(orders.loc[0:2]), "vs", len(orders.iloc[0:2]))

## Recap

- `Series` = array + index. `DataFrame` = dict of Series sharing one index.
- Arithmetic **aligns on the index**, not on position. Mismatched labels give NaN.
- `.loc` = labels (end **inclusive**), `.iloc` = positions (end exclusive). Never use bare `df[...]` for row selection except with a boolean mask.
- `& | ~` with parentheses for combined conditions; `isin`, `between`, `str.contains`, `query` cover the rest of `WHERE`.
- `int64` cannot hold nulls - it silently becomes `float64`. Use `Int64` when you need both.
- `category` dtype is free memory for repeated text.
- Prefer Parquet over CSV when you control the format.

**Next:** `05_pandas_groupby_join_reshape.ipynb` - `GROUP BY`, `JOIN`, and pivots.